<a href="https://colab.research.google.com/github/eliabrodsky/la_data/blob/main/Hospital_Analysis_VBC_EB_Sept_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Louisiana Rural Hospital Sustainability and VBC Readiness

Segments Louisiana's 64 rural hospitals on financial position and assesses which
are placed to enter value-based payment arrangements.

**Data:** CMS Hospital Provider Cost Reports (HCRIS, form CMS-2552-10), FY2021–FY2023.

---
# 1 &nbsp; Logic of the analysis

## The question

Which Louisiana rural hospitals can sustain themselves, what distinguishes them from
those that cannot, and which are placed to take on value-based care.

## What drives a rural hospital's financial position

Three things, in order of magnitude.

**Reimbursement class.** A Critical Access Hospital is paid close to cost. An IPPS
hospital receives a fixed price per case. That difference sets the shape of the
income statement before any management decision is made.

**Ownership structure.** A parish hospital district holds its own cash. A hospital
inside a corporate system has cash swept to the parent. The balance sheet reports the
treasury arrangement.

**Operating performance.** What remains once the first two are accounted for.

Value-based care participation depends on a fourth thing, independent of these:
control of primary care billing. Medicare attributes a patient to whoever bills their
primary care visits, so a hospital with no employed primary care providers brings no
attributable lives, whatever its size or solvency.

Separating these four is the work of the notebook. It is what keeps a well-reimbursed
hospital from reading as a strong operator, and a large hospital from reading as a VBC
candidate.

## How the analysis proceeds

| Section | Purpose |
|---|---|
| **2 Load** | Build a three-year panel and derive comparable ratios |
| **3 Explore** | Establish which metrics are comparable across hospitals |
| **4 Cluster** | Find groups that cut across existing categories, and test whether they hold |
| **5 Interpret** | Name what separates the groups and score VBC potential |

## Two properties of the data that shape the method

**Position moves year to year.** Segments assigned from a single year hold across three
years for about half of these hospitals. Cost-settled hospitals swing with settlement
timing. Features are therefore built as level, slope and volatility across all three
years, so trajectory is part of what the model sees.

**Structure contaminates the balance sheet.** Reimbursement class and ownership enter
as stratification variables. Comparisons are made within them, and they are kept out
of the clustering inputs.

## Definitions

| Term | Meaning here |
|---|---|
| **Rural hospital** | The 64-hospital universe with a Medicare cost report, from the CMS CAH list and the LDH rural designation lists |
| **Reimbursement class** | Medicare class: CAH, IPPS, SCH, RRC, REH |
| **Ownership stratum** | Type of Control from Worksheet S-2, collapsed to Government, Nonprofit, Proprietary |
| **Sustainability** | Ability to fund operations and reinvestment from recurring revenue, whether that revenue is patient care or policy |
| **VBC potential** | Risk-bearing capacity, operating performance, and attributable primary care lives. All three required |

---
# 2 &nbsp; Loading the data

## 2.1 &nbsp; Environment

Scientific stack plus scikit-learn. Plot defaults are set once so every figure
downstream renders consistently.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import kruskal, linregress
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import cdist

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold

warnings.filterwarnings('ignore', category=FutureWarning)

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

plt.rcParams.update({
    'figure.dpi': 110,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('Environment ready.')

## 2.2 &nbsp; Source files

Three annual CMS cost report public use files, read directly from GitHub so the
notebook runs from a clean Colab runtime.

`rural_ccn.csv` defines the 64-hospital rural universe and carries the LIHNC and Epic
pipeline flags. It also maps filing names to working names, since hospitals file under
legal entities: Prevost is West Ascension, Richland Parish Hospital Service District is
Richardson, Riverland is Trinity.

In [ ]:
BASE = 'https://raw.githubusercontent.com/eliabrodsky/la_data/main/'

FILES = {
    2021: BASE + 'CostReport_2021_Final.csv',
    2022: BASE + 'CostReport_2022_Final.csv',
    2023: BASE + 'CostReport_2023_Final.csv',
}

CROSSWALK = BASE + 'rural_ccn.csv'

probe = pd.read_table(FILES[2023], sep=',', header=0, nrows=5, low_memory=False)
print(f'Source reachable. {probe.shape[1]} columns per annual file.')
probe.head()

## 2.3 &nbsp; Filter to Louisiana and derive the ratio set

Each annual file covers every Medicare-certified hospital nationally. We keep Louisiana
and compute five ratios: liquidity, capital structure, short-term solvency, margin on
patient care, and margin on the bottom line.

Per-day ratios divide by the actual number of days in the reporting period. Cost reports
do not all cover a full year. A hospital that converts to Rural Emergency Hospital status,
changes its fiscal year end, or opens mid-year files a short period. Six Louisiana reports
cover under 330 days and one covers 32 days, where an annual denominator would report
5,140 days of cash on hand in place of 442.

In [ ]:
CONTROL = {
    1: 'Nonprofit-Church',       2: 'Nonprofit-Other',
    3: 'Proprietary-Individual', 4: 'Proprietary-Corp',
    5: 'Proprietary-Partnership', 6: 'Proprietary-Other',
    7: 'Gov-Federal',            8: 'Gov-City-County',
    9: 'Gov-County',            10: 'Gov-State',
    11: 'Gov-Hospital District', 12: 'Gov-City',
    13: 'Gov-Other',
}


def load_year(year, url):
    """Read one annual cost report file, keep Louisiana, derive the ratio set."""
    d = pd.read_table(url, sep=',', header=0, low_memory=False)
    d = d[d['State Code'] == 'LA'].copy()

    def col(name):
        if name not in d.columns:
            return pd.Series(np.nan, index=d.index)
        return pd.to_numeric(d[name], errors='coerce')

    begin  = pd.to_datetime(d['Fiscal Year Begin Date'], errors='coerce')
    end    = pd.to_datetime(d['Fiscal Year End Date'],   errors='coerce')
    period = (end - begin).dt.days.clip(lower=1)

    cash = col('Cash on Hand and in Banks').fillna(0) + col('Temporary Investments').fillna(0)
    opex = col('Less Total Operating Expense')
    dep  = col('Depreciation Cost').fillna(0)
    npr  = col('Net Patient Revenue')
    oth  = col('Total Other Income').fillna(0)

    out = pd.DataFrame({
        'fy':          year,
        'ccn':         d['Provider CCN'].astype(str).str.zfill(6),
        'hospital':    d['Hospital Name'].str.strip(),
        'city':        d['City'].str.strip().str.title(),
        'hcris_class': d['CCN Facility Type'],
        'control':     pd.to_numeric(d['Type of Control'], errors='coerce').map(CONTROL),
        'fy_days':     period,
        'npr':         npr,
    })

    # Liquidity: cash relative to daily cash operating expense, depreciation removed
    out['days_cash']     = (cash / ((opex - dep) / period)).where((opex - dep) > 0)

    # Capital structure
    out['equity_ratio']  = (col('Total Fund Balances') / col('Total Assets')
                            ).where(col('Total Assets') > 0)
    out['current_ratio'] = (col('Total Current Assets') / col('Total Current Liabilities')
                            ).where(col('Total Current Liabilities') > 0)

    # Profitability: on patient care alone, and on the bottom line
    out['pt_svc_margin'] = (col('Net Income from Service to Patients') / npr).where(npr > 0)
    out['total_margin']  = (col('Net Income') / (npr + oth)).where((npr + oth) > 0)

    print(f'  FY{year}: {len(out)} Louisiana hospitals')
    return out


print('Loading annual cost report files')
panel = pd.concat([load_year(y, u) for y, u in FILES.items()], ignore_index=True)

## 2.4 &nbsp; Restrict to the rural universe and assign ownership stratum

The 13 Type of Control codes collapse to three strata, used throughout as the unit
within which financial comparisons are made.

In [ ]:
xwalk = pd.read_table(CROSSWALK, sep=',', header=0, dtype=str)
xwalk['ccn'] = xwalk['ccn'].str.zfill(6)

panel = panel[panel['ccn'].isin(set(xwalk['ccn']))].copy()

panel['stratum'] = panel['control'].map(
    lambda c: 'Government'  if str(c).startswith('Gov')
    else     ('Proprietary' if str(c).startswith('Proprietary')
    else      'Nonprofit')
)

print(f"{len(panel)} hospital-years  |  {panel['ccn'].nunique()} hospitals  "
      f"|  FY{panel['fy'].min()}-FY{panel['fy'].max()}")
print()
print('Ownership stratum, most recent year')
print(panel[panel['fy'] == panel['fy'].max()]['stratum'].value_counts().to_string())

---
# 3 &nbsp; Exploratory analysis

This section establishes which metrics are comparable across hospitals. That determines
what the clustering model is allowed to contain.

## 3.1 &nbsp; Distribution by ownership

Days cash on hand and patient services margin, by ownership stratum.

In [ ]:
latest = panel[panel['fy'] == panel['fy'].max()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, metric, title in zip(
        axes,
        ['days_cash', 'pt_svc_margin'],
        ['Days cash on hand', 'Patient services margin']):

    order = ['Government', 'Nonprofit', 'Proprietary']
    data = [latest.loc[latest['stratum'] == s, metric].dropna() for s in order]
    ax.boxplot(data, tick_labels=order, showfliers=False)
    ax.set_title(title)
    ax.axhline(0, color='#c3c2b7', linewidth=0.8)

axes[0].set_ylabel('Days')
axes[1].set_ylabel('Share of net patient revenue')
plt.tight_layout()
plt.show()

print(latest.groupby('control')[['days_cash', 'equity_ratio', 'total_margin']]
      .agg(['size', 'median']).round(3)
      .sort_values(('days_cash', 'size'), ascending=False).to_string())

## 3.2 &nbsp; Confound test

Kruskal-Wallis tests whether distributions differ across groups without assuming
normality, which these skewed ratios violate. Two metrics against two groupings.

In [ ]:
print('Kruskal-Wallis tests')
print('-' * 66)

for metric in ['days_cash', 'pt_svc_margin']:
    for grouping in ['control', 'hcris_class']:
        groups = [g[metric].dropna().values
                  for _, g in latest.groupby(grouping)
                  if g[metric].notna().sum() >= 3]
        h, p = kruskal(*groups)

        if p < 0.01:
            verdict = 'CONFOUNDED'
        elif p < 0.10:
            verdict = 'borderline'
        else:
            verdict = 'clear'

        print(f'{metric:16s} ~ {grouping:12s}   H = {h:6.2f}   p = {p:.4f}   {verdict}')

### Result

**Days cash on hand differs by ownership** (p = 0.0001) and not by Medicare class
(p = 0.22).

- Government hospital districts hold their own cash: median around **120 days**
- Proprietary corporations sweep it to a parent: median around **8 days**, often with negative equity

That gap measures a treasury arrangement.

**Patient services margin is borderline on ownership** (p = 0.05) and clear on Medicare
class, so it holds as a direct cross-hospital comparison.

Liquidity therefore enters the model as a within-stratum percentile rank, measuring each
hospital against its own peer group. Operating margin enters as a raw value.

In [ ]:
for col in ['days_cash', 'equity_ratio']:
    panel[f'{col}_pct'] = panel.groupby(['fy', 'stratum'])[col].rank(pct=True)

print('Raw value vs within-stratum rank')
print(panel[['hospital', 'fy', 'stratum', 'days_cash', 'days_cash_pct']]
      .head(8).round(3).to_string(index=False))

## 3.3 &nbsp; Direction of travel

Each hospital's liquidity across the three years, with the sector median overlaid, and
the summary counts beneath it.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))

for _, g in panel.groupby('ccn'):
    if g['days_cash'].notna().sum() >= 2:
        ax.plot(g['fy'], g['days_cash'], color='gray', alpha=0.25, linewidth=0.8)

median = panel.groupby('fy')['days_cash'].median()
ax.plot(median.index, median.values, color='#1F3864', linewidth=2.5,
        marker='o', label='Median')

ax.set_ylim(0, 400)
ax.set_xticks(sorted(panel['fy'].unique()))
ax.set_ylabel('Days cash on hand')
ax.set_title('Liquidity trajectory, one line per hospital')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

trend = panel.groupby('fy').agg(
    n=('ccn', 'size'),
    days_cash=('days_cash', 'median'),
    pt_svc_margin=('pt_svc_margin', 'median'),
    total_margin=('total_margin', 'median'),
    under_60_days=('days_cash', lambda s: int((s < 60).sum())),
    negative_margin=('total_margin', lambda s: int((s < 0).sum())),
).round(3)

print(trend.to_string())

## 3.4 &nbsp; Hospital-level features

Reshape from one row per hospital-year to one row per hospital, with three views of
each metric.

- **Level** — where the hospital stands in the most recent year
- **Slope** — the direction it has moved across three years
- **Volatility** — how much it swings, which for cost-settled hospitals carries information

This is what distinguishes a hospital that is stable and weak from one that is
deteriorating.

In [ ]:
def slope(group, column):
    """Least-squares slope across available years. NaN if fewer than 2 points."""
    s = group.dropna(subset=[column])
    if len(s) < 2:
        return np.nan
    return linregress(s['fy'], s[column]).slope


METRICS = ['days_cash_pct', 'equity_ratio_pct', 'pt_svc_margin',
           'total_margin', 'days_cash', 'npr']

rows = []
for ccn, g in panel.groupby('ccn'):
    g = g.sort_values('fy')
    last = g.iloc[-1]

    row = {
        'ccn':         ccn,
        'hospital':    last['hospital'],
        'stratum':     last['stratum'],
        'control':     last['control'],
        'hcris_class': last['hcris_class'],
        'n_years':     len(g),
    }
    for m in METRICS:
        row[f'{m}_lvl']   = last[m]
        row[f'{m}_slope'] = slope(g, m)
        row[f'{m}_vol']   = g[m].std() if g[m].notna().sum() >= 2 else np.nan

    rows.append(row)

feat = pd.DataFrame(rows)
print(f'{len(feat)} hospitals, {feat.shape[1]} engineered columns')
feat.head()

## 3.5 &nbsp; Feature selection

With 64 hospitals, the model carries six features. The candidate list is longer, and
several candidates measure the same underlying quantity, so the correlation structure
decides which one represents each block: **liquidity**, **capital structure**,
**profitability**, **scale**.

Extreme values are winsorized at the 5th and 95th percentiles. The outliers here are
real hospitals.

In [ ]:
FEATURES = [
    'days_cash_pct_lvl',     # liquidity, ranked within ownership stratum
    'equity_ratio_pct_lvl',  # capital structure, ranked within stratum
    'pt_svc_margin_lvl',     # operating performance, where it stands
    'pt_svc_margin_slope',   # operating performance, where it is heading
    'total_margin_lvl',      # bottom line including non-patient revenue
    'npr_lvl',               # scale
]

X = feat[FEATURES].copy()
X['npr_lvl'] = np.log10(X['npr_lvl'].clip(lower=1))

complete = X.notna().all(axis=1)
Xc = X[complete].copy()

for c in Xc.columns:
    lo, hi = Xc[c].quantile([0.05, 0.95])
    Xc[c] = Xc[c].clip(lo, hi)

Xs = StandardScaler().fit_transform(Xc)
sub = feat[complete].reset_index(drop=True)

print(f'{complete.sum()} of {len(feat)} hospitals complete on all six features')
print()
print('Spearman correlation')
print(pd.DataFrame(Xs, columns=FEATURES).corr(method='spearman').round(2).to_string())

---
# 4 &nbsp; Clustering

## 4.1 &nbsp; Method

**Ward-linkage hierarchical clustering.** It is deterministic, produces a dendrogram
showing where the data splits, and requires no commitment to a number of groups in
advance. K-means runs alongside as a cross-check: agreement between the two indicates
real structure.

**Bootstrap stability** decides how many groups are reported. With 64 hospitals it is
easy to produce groups that dissolve when two hospitals are swapped. The procedure:

1. Resample hospitals with replacement and cluster the resample
2. Assign all original hospitals to the nearest resulting cluster centre
3. Measure Jaccard overlap between each original cluster and its best match
4. Repeat 400 times and average

Thresholds: **0.60 and above is stable**, **0.75 and above is highly stable**, below
0.50 indicates an artefact of the particular sample. A solution is reported only when
every cluster clears 0.60.

In [ ]:
def ward(X, k):
    """Ward-linkage flat clustering into k groups."""
    return fcluster(linkage(X, method='ward'), k, criterion='maxclust')


def clusterboot(X, k, n_boot=400, seed=0):
    """Bootstrap cluster stability. Returns base labels and mean Jaccard per cluster."""
    rng = np.random.default_rng(seed)
    base = ward(X, k)
    jaccard = np.zeros((n_boot, k))

    for b in range(n_boot):
        idx = rng.integers(0, len(X), len(X))
        Xb = X[idx]
        labels_b = ward(Xb, k)

        centroids = np.array([Xb[labels_b == c].mean(axis=0) for c in range(1, k + 1)])
        assigned = np.argmin(cdist(X, centroids), axis=1) + 1

        for c in range(1, k + 1):
            original = set(np.where(base == c)[0])
            best = 0.0
            for rc in range(1, k + 1):
                resampled = set(np.where(assigned == rc)[0])
                union = len(original | resampled)
                if union:
                    best = max(best, len(original & resampled) / union)
            jaccard[b, c - 1] = best

    return base, jaccard.mean(axis=0)

## 4.2 &nbsp; Number of groups

Three diagnostics, read together. Silhouette measures separation, where 0.2 to 0.3 is
weak but present. ARI measures whether two algorithms find the same groups. Bootstrap
Jaccard measures survival under resampling, and decides the answer.

In [ ]:
print(f"{'k':>2}  {'silhouette':>10}  {'ARI vs kmeans':>13}   cluster sizes and stability")
print('-' * 90)

for k in range(2, 6):
    base, jac = clusterboot(Xs, k)
    km = KMeans(n_clusters=k, n_init=25, random_state=0).fit(Xs)
    sizes = np.bincount(base)[1:]

    detail = '   '.join(f'n={s:<3d} J={j:.2f}' for s, j in zip(sizes, jac))
    print(f'{k:>2}  {silhouette_score(Xs, base):>10.3f}  '
          f'{adjusted_rand_score(base, km.labels_):>13.2f}   '
          f'{detail}   mean J = {jac.mean():.3f}')

### Result

**Two groups**, stable at roughly 0.65 and 0.73.

From k = 3 upward, a four-hospital cluster appears with a Jaccard near 0.38, placing it
in artefact territory. At k = 4 only the largest cluster clears the threshold and mean
stability falls to 0.53. Silhouettes run 0.20 to 0.24 across the range and the two
algorithms agree only moderately, both consistent with a single split rather than a
graded set of tiers.

In [ ]:
K = 2
sub['cluster'] = ward(Xs, K)

fig, ax = plt.subplots(figsize=(13, 4.5))
dendrogram(linkage(Xs, method='ward'),
           labels=sub['hospital'].values,
           leaf_rotation=90,
           leaf_font_size=6,
           ax=ax)
ax.set_title('Ward linkage, Louisiana rural hospitals')
ax.set_ylabel('Distance')
plt.tight_layout()
plt.show()

---
# 5 &nbsp; Interpretation

## 5.1 &nbsp; Profile of the two groups

In [ ]:
profile_cols = ['days_cash_pct_lvl', 'equity_ratio_pct_lvl', 'pt_svc_margin_lvl',
                'pt_svc_margin_slope', 'total_margin_lvl', 'days_cash_lvl']

print('Cluster profile, medians')
print(sub.groupby('cluster')[profile_cols].median().round(3).to_string())
print()
print('Sizes:', np.bincount(sub['cluster'])[1:])

| | Cluster 1 (n = 27) | Cluster 2 (n = 33) |
|---|---|---|
| Patient services margin | −30% | −8% |
| Direction of travel | worsening | improving |
| Total margin | −4% | +12% |
| Days cash on hand | 39 | 108 |
| Equity percentile within stratum | 0.33 | 0.69 |

Both groups lose money delivering care, which is the standard condition for a Louisiana
rural hospital. The split is the size of the gap and whether it is closing or widening.

## 5.2 &nbsp; What separates them

A depth-two decision tree gives the split rule. The cross-tabulations show how the
clusters sit against categories already present in the data.

In [ ]:
tree = DecisionTreeClassifier(max_depth=2, random_state=0).fit(Xs, sub['cluster'])
print('Split rule')
print(export_text(tree, feature_names=FEATURES))

for col in ['hcris_class', 'stratum']:
    print(f'Cluster by {col}')
    print(pd.crosstab(sub['cluster'], sub[col]).to_string())
    print()

The split runs on total margin and equity rank, and spreads across both Medicare class
and ownership stratum. It is a new grouping, not a restatement of CAH status or of who
owns the hospital.

## 5.3 &nbsp; Do existing program groupings track financial position?

Take each grouping already in use and predict membership from the financial features
alone. Accuracy at or below the base rate means the grouping carries no information
about financial position.

In [ ]:
flags = xwalk[['ccn', 'lihnc_member', 'in_epic_pipeline']]
sub = sub.merge(flags, on='ccn', how='left')

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)

targets = {
    'CAH designation': (sub['hcris_class'] == 'CAH').astype(int),
    'LIHNC member':    (sub['lihnc_member'] == 'Yes').astype(int),
    'Epic pipeline':   (sub['in_epic_pipeline'] == 'Yes').astype(int),
}

print(f"{'Grouping':<18}{'base rate':>11}{'CV accuracy':>13}{'lift':>8}   verdict")
print('-' * 72)

for name, y in targets.items():
    base_rate = max(y.mean(), 1 - y.mean())
    acc = cross_val_score(LogisticRegression(C=0.5, max_iter=2000),
                          Xs, y, cv=cv, scoring='accuracy').mean()
    lift = acc - base_rate
    verdict = 'carries signal' if lift > 0.05 else 'no financial signal'
    print(f'{name:<18}{base_rate:>11.2f}{acc:>13.2f}{lift:>+8.2f}   {verdict}')

### Result

**CAH designation is predictable** from financial position, as reimbursement class and
finances are linked.

**LIHNC membership is not. Epic pipeline participation is not.** Both score below the
base rate, so the financial features perform worse than guessing the majority class.

Financial position played no part in determining who joined the network or who entered
the Epic waves. Two consequences follow. Neither grouping works as a proxy for readiness.
And if either program is meant to reach hospitals under financial pressure, it is not
currently selecting for that.

## 5.4 &nbsp; Scoring VBC potential

Section 1 defined VBC potential as three requirements. Two are measurable from cost
report data. The third is measurable from no available dataset and is carried as a blank.

| Axis | Source | Status |
|---|---|---|
| **Risk-bearing capacity** | Liquidity and equity, ranked within ownership stratum | Scored |
| **Operating performance** | Patient services margin, its trend, and total margin | Scored |
| **Attribution capacity** | Primary care providers billing under the hospital's own TIN, and certified provider-based RHCs | **No data. Requires a survey** |

The axes stay separate. A hospital strong on both scored axes and empty on the third is
not a candidate, and a single combined score would conceal that case.

In [ ]:
z = pd.DataFrame(Xs, columns=FEATURES)

sub['risk_capacity']  = z[['days_cash_pct_lvl', 'equity_ratio_pct_lvl']].mean(axis=1)
sub['operating_perf'] = z[['pt_svc_margin_lvl', 'pt_svc_margin_slope',
                           'total_margin_lvl']].mean(axis=1)
sub['attribution']    = np.nan   # awaiting the primary care and billing TIN survey

for c in ['risk_capacity', 'operating_perf']:
    sub[f'{c}_pct'] = sub[c].rank(pct=True).round(2)

fig, ax = plt.subplots(figsize=(7.5, 6))

palette = {1: ('#E24B4A', 'Cluster 1  (n=27)'),
           2: ('#1D9E75', 'Cluster 2  (n=33)')}

for cl, (colour, label) in palette.items():
    s = sub[sub['cluster'] == cl]
    ax.scatter(s['operating_perf'], s['risk_capacity'], c=colour, s=48,
               alpha=0.75, edgecolor='white', linewidth=0.5, label=label)

ax.axhline(0, color='#c3c2b7', linewidth=0.8)
ax.axvline(0, color='#c3c2b7', linewidth=0.8)
ax.set_xlabel('Operating performance  (standardized)')
ax.set_ylabel('Risk-bearing capacity  (within ownership stratum)')
ax.set_title('Two scored axes. The third, attribution, has no data.')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
scorecard = (sub[['hospital', 'hcris_class', 'stratum', 'cluster',
                  'risk_capacity_pct', 'operating_perf_pct', 'attribution',
                  'days_cash_lvl', 'pt_svc_margin_lvl', 'total_margin_lvl']]
             .sort_values('operating_perf_pct', ascending=False)
             .round(3))

scorecard.to_csv('la_hospital_scorecard.csv', index=False)
print(f'Wrote la_hospital_scorecard.csv  ({len(scorecard)} hospitals)')

# In Colab: from google.colab import files; files.download('la_hospital_scorecard.csv')

scorecard.head(15)

## 5.5 &nbsp; What the analysis does not answer

1. **Cluster membership is provisional.** Stability of 0.65 and 0.73 supports a
   two-group split at the sector level. A single hospital's position should be checked
   against its own three-year record before it is used in conversation.

2. **There is no attribution measure.** The count of primary care providers billing
   under each hospital's own TIN, and of certified provider-based rural health clinics,
   appears in no public dataset. A short survey of the membership supplies it, and it is
   the decisive missing input for the VBC question.

3. **There is no quality measure.** Financial capacity is one of two criteria for
   selecting VBC participants. Care Compare and MIPS results belong here before any
   candidate list is published.

4. **Non-patient revenue is not decomposed.** Total Other Income is a single line, so ad
   valorem tax, cost settlement, grants and investment income cannot be separated. For a
   parish hospital district much of the gap between patient services margin and total
   margin may be property tax rather than Medicaid policy, and the two carry different
   outlooks. Worksheet G-3 from the full HCRIS extract separates the tax component.

5. **Nothing here is causal.** 64 hospitals over three years with no exogenous variation
   supports description.

6. **Coverage.** Four hospitals are excluded for incomplete features and two have fewer
   than three years of filings.